# Sentiment Analysis — Model Comparison on Best Domain (DVD)

**Best domain from 4-Fold experiment: DVD (88.60%)**

This notebook compares 4 models on the same train/test split:

| Model | Description |
|---|---|
| BERT | Full BERT base model (110M params) |
| DistilBERT | Lightweight BERT, 40% faster |
| DistilBERT + DANN | DistilBERT with Domain Adaptation |
| BERT + DANN | Full BERT with Domain Adaptation |

**Train:** Books + Electronics + Kitchen → **Test:** DVD


## Step 1 — Install Dependencies

In [1]:
!pip install -q transformers torch nltk scikit-learn


## Step 2 — Import Libraries

In [2]:
import os, re, nltk, urllib.request, tarfile
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from transformers import (
    BertTokenizer, BertModel,
    DistilBertTokenizer, DistilBertModel,
    pipeline
)

nltk.download('stopwords')
nltk.download('punkt')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Libraries imported!')
print(f'Device: {device}')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Libraries imported!
Device: cuda


## Step 3 — Download Dataset

In [3]:
url          = 'https://www.cs.jhu.edu/~mdredze/datasets/sentiment/domain_sentiment_data.tar.gz'
tar_path     = 'domain_sentiment_data.tar.gz'
extract_path = 'sentiment_data'

if not os.path.exists(extract_path):
    print('Downloading dataset (~30MB)...')
    urllib.request.urlretrieve(url, tar_path)
    print('Extracting...')
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(extract_path, filter='data')
    print('Done!')
else:
    print('Dataset already downloaded.')

path = f'{extract_path}/sorted_data_acl/'
print('Folders:', os.listdir(path))


Extracting...
Done!
Folders: ['books', 'electronics', 'kitchen_&_housewares', 'dvd']


## Step 4 — Text Cleaning & Data Loading

In [4]:
def clean_sentence(sentence: str) -> str:
    sentence = re.sub(r'(<review_text>|<\/review_text>)', '', sentence)
    sentence = sentence.lower()
    sentence = re.sub(r'\bhttp\S+|\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', '', sentence)
    sentence = re.sub(r'@', 'a', sentence)
    sentence = re.sub(r'[^\w\s\-]', '', sentence)
    sentence = re.sub(r'\s+', ' ', sentence).strip()
    return sentence

regex_review = re.compile(r'.+?<\/review_text>', flags=re.DOTALL)

def read_reviews(folders, path, samples_per_class=500):
    x, y = [], []
    for folder in folders:
        for label, fname in [(0, 'negative.review'), (1, 'positive.review')]:
            raw     = open(path + folder + '/' + fname, 'r', encoding='utf-8', errors='ignore').read()
            reviews = re.findall(regex_review, raw)[:samples_per_class]
            sentiment = 'Negative' if label == 0 else 'Positive'
            print(f'  {len(reviews)} {sentiment} from [{folder}]')
            for s in reviews:
                x.append(clean_sentence(s))
                y.append(label)
    return x, y

# DVD is best domain — use it as test
# Train = Books + Electronics + Kitchen
TRAIN_DOMAINS = ['books', 'electronics', 'kitchen_&_housewares']
TEST_DOMAIN   = 'dvd'

print('Loading Train Data (Books + Electronics + Kitchen)...')
x_train, y_train = read_reviews(TRAIN_DOMAINS, path, samples_per_class=300)

print('\nLoading Test Data (DVD)...')
x_test, y_test = read_reviews([TEST_DOMAIN], path, samples_per_class=500)

print(f'\nTrain samples : {len(x_train)}')
print(f'Test samples  : {len(x_test)}')


Loading Train Data (Books + Electronics + Kitchen)...
  300 Negative from [books]
  300 Positive from [books]
  300 Negative from [electronics]
  300 Positive from [electronics]
  300 Negative from [kitchen_&_housewares]
  300 Positive from [kitchen_&_housewares]

Loading Test Data (DVD)...
  500 Negative from [dvd]
  500 Positive from [dvd]

Train samples : 1800
Test samples  : 1000


## Step 5 — Results Printer

In [5]:
all_results = {}  # stores accuracy of each model for final comparison

def print_results(model_name, y_true, y_pred):
    acc    = accuracy_score(y_true, y_pred)
    cm     = confusion_matrix(y_true, y_pred)
    report = classification_report(
        y_true, y_pred,
        labels=[0, 1],
        target_names=['Negative', 'Positive'],
        zero_division=0
    )
    print('\n' + '=' * 60)
    print(f'  {model_name} RESULTS')
    print('=' * 60)
    print(f'  Test Domain : DVD')
    print(f'  Accuracy    : {acc * 100:.2f}%')
    print('=' * 60)
    print('\nClassification Report:')
    print(report)
    print('Confusion Matrix:')
    print(cm)
    print(f'  True Negatives  : {cm[0][0]}')
    print(f'  False Positives : {cm[0][1]}')
    print(f'  False Negatives : {cm[1][0]}')
    print(f'  True Positives  : {cm[1][1]}')
    print('=' * 60)
    all_results[model_name] = acc
    return acc


---
## Model 1 — BERT (bert-base-uncased-SST-2)

Full BERT model, 110M parameters, fine-tuned on SST-2 sentiment dataset.


In [6]:
print('Loading BERT model (~440MB)...')
bert_pipeline = pipeline(
    'sentiment-analysis',
    model='textattack/bert-base-uncased-SST-2',
    truncation=True, max_length=512,
    device=0 if torch.cuda.is_available() else -1
)

def bert_predict(text):
    if not text.strip(): return 0
    label = bert_pipeline(text)[0]['label'].upper()
    if label == 'LABEL_1': return 1
    if label == 'LABEL_0': return 0
    return 1 if 'POS' in label else 0

print('Running BERT predictions on DVD test set...')
y_pred_bert = []
for i, review in enumerate(x_test):
    y_pred_bert.append(bert_predict(review))
    if (i+1) % 100 == 0 or (i+1) == len(x_test):
        print(f'  [{i+1}/{len(x_test)}] processed...')

acc_bert = print_results('BERT', y_test, y_pred_bert)


Loading BERT model (~440MB)...


config.json:   0%|          | 0.00/477 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Running BERT predictions on DVD test set...


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  [100/1000] processed...
  [200/1000] processed...
  [300/1000] processed...
  [400/1000] processed...
  [500/1000] processed...
  [600/1000] processed...
  [700/1000] processed...
  [800/1000] processed...
  [900/1000] processed...
  [1000/1000] processed...

  BERT RESULTS
  Test Domain : DVD
  Accuracy    : 88.60%

Classification Report:
              precision    recall  f1-score   support

    Negative       0.85      0.94      0.89       500
    Positive       0.94      0.83      0.88       500

    accuracy                           0.89      1000
   macro avg       0.89      0.89      0.89      1000
weighted avg       0.89      0.89      0.89      1000

Confusion Matrix:
[[472  28]
 [ 86 414]]
  True Negatives  : 472
  False Positives : 28
  False Negatives : 86
  True Positives  : 414


---
## Model 2 — DistilBERT (distilbert-base-uncased-finetuned-sst-2-english)

Lightweight BERT — 66M parameters, 40% faster, ~97% of BERT accuracy.


In [7]:
print('Loading DistilBERT model (~268MB)...')
distilbert_pipeline = pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english',
    truncation=True, max_length=512,
    device=0 if torch.cuda.is_available() else -1
)

def distilbert_predict(text):
    if not text.strip(): return 0
    label = distilbert_pipeline(text)[0]['label'].upper()
    if label == 'LABEL_1': return 1
    if label == 'LABEL_0': return 0
    return 1 if 'POS' in label else 0

print('Running DistilBERT predictions on DVD test set...')
y_pred_distilbert = []
for i, review in enumerate(x_test):
    y_pred_distilbert.append(distilbert_predict(review))
    if (i+1) % 100 == 0 or (i+1) == len(x_test):
        print(f'  [{i+1}/{len(x_test)}] processed...')

acc_distilbert = print_results('DistilBERT', y_test, y_pred_distilbert)


Loading DistilBERT model (~268MB)...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Running DistilBERT predictions on DVD test set...
  [100/1000] processed...
  [200/1000] processed...
  [300/1000] processed...
  [400/1000] processed...
  [500/1000] processed...
  [600/1000] processed...
  [700/1000] processed...
  [800/1000] processed...
  [900/1000] processed...
  [1000/1000] processed...

  DistilBERT RESULTS
  Test Domain : DVD
  Accuracy    : 84.80%

Classification Report:
              precision    recall  f1-score   support

    Negative       0.79      0.96      0.86       500
    Positive       0.94      0.74      0.83       500

    accuracy                           0.85      1000
   macro avg       0.87      0.85      0.85      1000
weighted avg       0.87      0.85      0.85      1000

Confusion Matrix:
[[478  22]
 [130 370]]
  True Negatives  : 478
  False Positives : 22
  False Negatives : 130
  True Positives  : 370


---
## Model 3 — DistilBERT + DANN

**DANN = Domain Adversarial Neural Network**

DANN adds a domain classifier on top of DistilBERT that tries to make the model
**domain-agnostic** — it learns features that work across Books, Electronics, Kitchen
AND DVD, rather than overfitting to one domain's writing style.

Architecture:
```
Input Review
     ↓
DistilBERT (feature extractor)
     ↓
  [CLS] embedding (768-dim)
     ↓                    ↓
Sentiment Classifier   Domain Classifier (with gradient reversal)
(Positive/Negative)    (Books/DVD/Electronics/Kitchen)
```
The **Gradient Reversal Layer** forces the feature extractor to produce embeddings
that fool the domain classifier — meaning the features become domain-independent.


In [8]:
from transformers import DistilBertTokenizer, DistilBertModel

# ── Gradient Reversal Layer ──────────────────────────────
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class GradientReversalLayer(nn.Module):
    def __init__(self, alpha=1.0):
        super().__init__()
        self.alpha = alpha
    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

# ── DistilBERT + DANN Model ───────────────────────────────
class DistilBertDANN(nn.Module):
    def __init__(self, num_domains=4, alpha=1.0):
        super().__init__()
        self.distilbert   = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.grl          = GradientReversalLayer(alpha)
        # Sentiment classifier head
        self.sentiment_classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )
        # Domain classifier head
        self.domain_classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_domains)
        )

    def forward(self, input_ids, attention_mask):
        outputs    = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]   # [CLS] token
        sentiment_logits = self.sentiment_classifier(cls_output)
        domain_logits    = self.domain_classifier(self.grl(cls_output))
        return sentiment_logits, domain_logits

print('DistilBERT+DANN model class defined!')


DistilBERT+DANN model class defined!


In [9]:
# ── Dataset for DANN ──────────────────────────────────────
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, domain_labels, tokenizer, max_len=128):
        self.texts        = texts
        self.labels       = labels
        self.domain_labels= domain_labels
        self.tokenizer    = tokenizer
        self.max_len      = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'     : enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label'         : torch.tensor(self.labels[idx], dtype=torch.long),
            'domain_label'  : torch.tensor(self.domain_labels[idx], dtype=torch.long)
        }

# Assign domain IDs to train data
# books=0, electronics=1, kitchen=2  (each has 300 neg + 300 pos = 600 reviews)
domain_ids = ([0]*600 + [1]*600 + [2]*600)

tokenizer_distil = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

train_dataset = ReviewDataset(x_train, y_train, domain_ids, tokenizer_distil)
train_loader  = DataLoader(train_dataset, batch_size=16, shuffle=True)

print(f'Train dataset size : {len(train_dataset)}')
print(f'Batches per epoch  : {len(train_loader)}')


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Train dataset size : 1800
Batches per epoch  : 113


In [10]:
# ── Train DistilBERT + DANN ───────────────────────────────
EPOCHS = 3

dann_distil = DistilBertDANN(num_domains=3, alpha=1.0).to(device)
optimizer   = optim.AdamW(dann_distil.parameters(), lr=2e-5)
sentiment_loss_fn = nn.CrossEntropyLoss()
domain_loss_fn    = nn.CrossEntropyLoss()

print(f'Training DistilBERT+DANN for {EPOCHS} epochs...')
print('(This may take 10-15 minutes on GPU)\n')

for epoch in range(EPOCHS):
    dann_distil.train()
    total_loss = 0
    correct    = 0
    total      = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)
        domain_labels  = batch['domain_label'].to(device)

        optimizer.zero_grad()
        sentiment_logits, domain_logits = dann_distil(input_ids, attention_mask)

        s_loss = sentiment_loss_fn(sentiment_logits, labels)
        d_loss = domain_loss_fn(domain_logits, domain_labels)
        loss   = s_loss + 0.1 * d_loss   # domain loss weighted at 0.1

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds       = sentiment_logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

        if (batch_idx + 1) % 30 == 0:
            print(f'  Epoch {epoch+1} | Batch {batch_idx+1}/{len(train_loader)} | '
                  f'Loss: {total_loss/(batch_idx+1):.4f} | '
                  f'Train Acc: {correct/total*100:.1f}%')

    print(f'\nEpoch {epoch+1} complete | Avg Loss: {total_loss/len(train_loader):.4f} | '
          f'Train Accuracy: {correct/total*100:.2f}%\n')

print('Training complete!')


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Training DistilBERT+DANN for 3 epochs...
(This may take 10-15 minutes on GPU)

  Epoch 1 | Batch 30/113 | Loss: 0.8094 | Train Acc: 50.6%
  Epoch 1 | Batch 60/113 | Loss: 0.7627 | Train Acc: 58.9%
  Epoch 1 | Batch 90/113 | Loss: 0.5765 | Train Acc: 72.6%

Epoch 1 complete | Avg Loss: 0.4851 | Train Accuracy: 78.06%

  Epoch 2 | Batch 30/113 | Loss: 0.1239 | Train Acc: 100.0%
  Epoch 2 | Batch 60/113 | Loss: 0.1353 | Train Acc: 99.7%
  Epoch 2 | Batch 90/113 | Loss: 0.1313 | Train Acc: 99.8%

Epoch 2 complete | Avg Loss: 0.1303 | Train Accuracy: 99.83%

  Epoch 3 | Batch 30/113 | Loss: 0.1373 | Train Acc: 100.0%
  Epoch 3 | Batch 60/113 | Loss: 0.1441 | Train Acc: 100.0%
  Epoch 3 | Batch 90/113 | Loss: 0.1457 | Train Acc: 100.0%

Epoch 3 complete | Avg Loss: 0.1462 | Train Accuracy: 100.00%

Training complete!


In [11]:
# ── Evaluate DistilBERT + DANN ────────────────────────────
dann_distil.eval()

test_dataset = ReviewDataset(x_test, y_test, [0]*len(y_test), tokenizer_distil)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

y_pred_dann_distil = []
print('Evaluating DistilBERT+DANN on DVD test set...')

with torch.no_grad():
    for i, batch in enumerate(test_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        sentiment_logits, _ = dann_distil(input_ids, attention_mask)
        preds = sentiment_logits.argmax(dim=1).cpu().numpy()
        y_pred_dann_distil.extend(preds)
        if (i+1) % 10 == 0 or (i+1) == len(test_loader):
            print(f'  [{(i+1)*32}/{len(x_test)}] processed...')

acc_dann_distil = print_results('DistilBERT + DANN', y_test, y_pred_dann_distil)


Evaluating DistilBERT+DANN on DVD test set...
  [320/1000] processed...
  [640/1000] processed...
  [960/1000] processed...
  [1024/1000] processed...

  DistilBERT + DANN RESULTS
  Test Domain : DVD
  Accuracy    : 68.00%

Classification Report:
              precision    recall  f1-score   support

    Negative       0.61      1.00      0.76       500
    Positive       1.00      0.36      0.53       500

    accuracy                           0.68      1000
   macro avg       0.80      0.68      0.64      1000
weighted avg       0.80      0.68      0.64      1000

Confusion Matrix:
[[500   0]
 [320 180]]
  True Negatives  : 500
  False Positives : 0
  False Negatives : 320
  True Positives  : 180


---
## Model 4 — BERT + DANN

Same DANN architecture but using **full BERT** (bert-base-uncased) as the feature extractor
instead of DistilBERT. More powerful but slower to train.

```
Input Review
     ↓
BERT base uncased (feature extractor)
     ↓
  [CLS] embedding (768-dim)
     ↓                    ↓
Sentiment Classifier   Domain Classifier (with gradient reversal)
```


In [12]:
from transformers import BertTokenizer, BertModel

class BertDANN(nn.Module):
    def __init__(self, num_domains=3, alpha=1.0):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.grl  = GradientReversalLayer(alpha)
        self.sentiment_classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )
        self.domain_classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_domains)
        )

    def forward(self, input_ids, attention_mask):
        outputs    = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.pooler_output   # BERT uses pooler output for [CLS]
        sentiment_logits = self.sentiment_classifier(cls_output)
        domain_logits    = self.domain_classifier(self.grl(cls_output))
        return sentiment_logits, domain_logits

print('BERT+DANN model class defined!')


BERT+DANN model class defined!


In [13]:
# ── Dataset for BERT+DANN ─────────────────────────────────
tokenizer_bert = BertTokenizer.from_pretrained('bert-base-uncased')

train_dataset_bert = ReviewDataset(x_train, y_train, domain_ids, tokenizer_bert)
train_loader_bert  = DataLoader(train_dataset_bert, batch_size=16, shuffle=True)

print(f'Train dataset size : {len(train_dataset_bert)}')


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Train dataset size : 1800


In [14]:
# ── Train BERT + DANN ─────────────────────────────────────
EPOCHS = 3

dann_bert  = BertDANN(num_domains=3, alpha=1.0).to(device)
optimizer2 = optim.AdamW(dann_bert.parameters(), lr=2e-5)

print(f'Training BERT+DANN for {EPOCHS} epochs...')
print('(This may take 15-20 minutes on GPU)\n')

for epoch in range(EPOCHS):
    dann_bert.train()
    total_loss = 0
    correct    = 0
    total      = 0

    for batch_idx, batch in enumerate(train_loader_bert):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)
        domain_labels  = batch['domain_label'].to(device)

        optimizer2.zero_grad()
        sentiment_logits, domain_logits = dann_bert(input_ids, attention_mask)

        s_loss = sentiment_loss_fn(sentiment_logits, labels)
        d_loss = domain_loss_fn(domain_logits, domain_labels)
        loss   = s_loss + 0.1 * d_loss

        loss.backward()
        optimizer2.step()

        total_loss += loss.item()
        preds       = sentiment_logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

        if (batch_idx + 1) % 30 == 0:
            print(f'  Epoch {epoch+1} | Batch {batch_idx+1}/{len(train_loader_bert)} | '
                  f'Loss: {total_loss/(batch_idx+1):.4f} | '
                  f'Train Acc: {correct/total*100:.1f}%')

    print(f'\nEpoch {epoch+1} complete | Avg Loss: {total_loss/len(train_loader_bert):.4f} | '
          f'Train Accuracy: {correct/total*100:.2f}%\n')

print('Training complete!')


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Training BERT+DANN for 3 epochs...
(This may take 15-20 minutes on GPU)

  Epoch 1 | Batch 30/113 | Loss: 0.8058 | Train Acc: 50.6%
  Epoch 1 | Batch 60/113 | Loss: 0.7727 | Train Acc: 59.8%
  Epoch 1 | Batch 90/113 | Loss: 0.6182 | Train Acc: 72.8%

Epoch 1 complete | Avg Loss: 0.5238 | Train Accuracy: 78.22%

  Epoch 2 | Batch 30/113 | Loss: 0.1313 | Train Acc: 100.0%
  Epoch 2 | Batch 60/113 | Loss: 0.1272 | Train Acc: 100.0%
  Epoch 2 | Batch 90/113 | Loss: 0.1248 | Train Acc: 100.0%

Epoch 2 complete | Avg Loss: 0.1235 | Train Accuracy: 100.00%

  Epoch 3 | Batch 30/113 | Loss: 0.1164 | Train Acc: 100.0%
  Epoch 3 | Batch 60/113 | Loss: 0.1161 | Train Acc: 100.0%
  Epoch 3 | Batch 90/113 | Loss: 0.1159 | Train Acc: 100.0%

Epoch 3 complete | Avg Loss: 0.1157 | Train Accuracy: 100.00%

Training complete!


In [15]:
# ── Evaluate BERT + DANN ──────────────────────────────────
dann_bert.eval()

test_dataset_bert = ReviewDataset(x_test, y_test, [0]*len(y_test), tokenizer_bert)
test_loader_bert  = DataLoader(test_dataset_bert, batch_size=32, shuffle=False)

y_pred_dann_bert = []
print('Evaluating BERT+DANN on DVD test set...')

with torch.no_grad():
    for i, batch in enumerate(test_loader_bert):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        sentiment_logits, _ = dann_bert(input_ids, attention_mask)
        preds = sentiment_logits.argmax(dim=1).cpu().numpy()
        y_pred_dann_bert.extend(preds)
        if (i+1) % 10 == 0 or (i+1) == len(test_loader_bert):
            print(f'  [{(i+1)*32}/{len(x_test)}] processed...')

acc_dann_bert = print_results('BERT + DANN', y_test, y_pred_dann_bert)


Evaluating BERT+DANN on DVD test set...
  [320/1000] processed...
  [640/1000] processed...
  [960/1000] processed...
  [1024/1000] processed...

  BERT + DANN RESULTS
  Test Domain : DVD
  Accuracy    : 67.70%

Classification Report:
              precision    recall  f1-score   support

    Negative       0.61      1.00      0.76       500
    Positive       1.00      0.35      0.52       500

    accuracy                           0.68      1000
   macro avg       0.80      0.68      0.64      1000
weighted avg       0.80      0.68      0.64      1000

Confusion Matrix:
[[500   0]
 [323 177]]
  True Negatives  : 500
  False Positives : 0
  False Negatives : 323
  True Positives  : 177


---
## Final Comparison — All 4 Models on DVD Domain

In [16]:
print('\n' + '=' * 65)
print('  FINAL MODEL COMPARISON — Test Domain: DVD')
print('=' * 65)
print(f'  {"Model":<25} {"Accuracy":>10}  Bar')
print('-' * 65)

for model_name, acc in all_results.items():
    bar = '█' * int(acc * 30)
    print(f'  {model_name:<25} {acc*100:>9.2f}%  {bar}')

print('=' * 65)
best_model = max(all_results, key=all_results.get)
best_acc   = all_results[best_model]
print(f'  Best Model  : {best_model} ({best_acc*100:.2f}%)')
print('=' * 65)

# Improvement of DANN over base models
if 'DistilBERT' in all_results and 'DistilBERT + DANN' in all_results:
    diff = (all_results['DistilBERT + DANN'] - all_results['DistilBERT']) * 100
    print(f'\n  DANN improvement over DistilBERT : {diff:+.2f}%')
if 'BERT' in all_results and 'BERT + DANN' in all_results:
    diff = (all_results['BERT + DANN'] - all_results['BERT']) * 100
    print(f'  DANN improvement over BERT       : {diff:+.2f}%')



  FINAL MODEL COMPARISON — Test Domain: DVD
  Model                       Accuracy  Bar
-----------------------------------------------------------------
  BERT                          88.60%  ██████████████████████████
  DistilBERT                    84.80%  █████████████████████████
  DistilBERT + DANN             68.00%  ████████████████████
  BERT + DANN                   67.70%  ████████████████████
  Best Model  : BERT (88.60%)

  DANN improvement over DistilBERT : -16.80%
  DANN improvement over BERT       : -20.90%


## Live Demo — Best Model Prediction

In [17]:
def predict_best(review: str):
    """Uses the best performing model for prediction."""
    cleaned = clean_sentence(review)
    # Use BERT pipeline (zero-shot, always available)
    label   = bert_pipeline(cleaned)[0]['label'].upper()
    if label == 'LABEL_1': result = 1
    elif label == 'LABEL_0': result = 0
    else: result = 1 if 'POS' in label else 0
    emoji = '✅ Positive Review' if result == 1 else '❌ Negative Review'
    print(f'Review : {review[:70]}')
    print(f'Result : {emoji}')
    print()

print('--- Live Demo ---\n')
predict_best('This dvd is absolutely fantastic, best purchase ever')
predict_best('Terrible quality, the dvd stopped working after one use')
predict_best('Great movie, loved every minute of it')
predict_best('Complete waste of money, very disappointed')


--- Live Demo ---

Review : This dvd is absolutely fantastic, best purchase ever
Result : ✅ Positive Review

Review : Terrible quality, the dvd stopped working after one use
Result : ❌ Negative Review

Review : Great movie, loved every minute of it
Result : ✅ Positive Review

Review : Complete waste of money, very disappointed
Result : ❌ Negative Review

